# Challenge 08 — โมเดลทำนายมูลค่านักเตะ (Market Value Prediction)

**Week 08 · Machine Learning เบื้องต้น · 30 นาที**

## ภารกิจ
1. ใช้สถิตินักเตะ เช่น `overall`, `age`, `pace`, `shooting`, `passing` เป็น **Feature ($X$)** เพื่อทำนาย `value_millions` (มูลค่านักเตะหน่วยล้าน) ซึ่งเป็น **Target ($y$)**
2. ทำความสะอาดข้อมูล `value_millions` และแปลงเป็นตัวเลข
3. แบ่งข้อมูลเป็น Train set (80%) และ Test set (20%)
4. เทรนโมเดล `LinearRegression` และวัดผลด้วย MAE (Mean Absolute Error) เทียบกับ Baseline
5. **ทำนายข้อมูลชุดใหม่ (Unseen Data):** อ่านไฟล์ `../data/scouted_players.csv` แล้วใช้โมเดลทำนายมูลค่าของนักเตะดาวรุ่งชุดใหม่ที่ไม่เคยอยู่ในข้อมูลเดิม!

## Debug checkpoint
- `X` ต้องเป็นตาราง 2 มิติ (DataFrame) จึงใช้วงเล็บซ้อน `df[['col1', 'col2', ...]]`
- ข้อมูล `value_millions` อาจมีค่าที่ไม่ใช่ตัวเลข (เช่น `'unknown'`) ต้องแปลงด้วย `pd.to_numeric(errors='coerce')` แล้วตัดแถวว่างออกก่อน

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# 1. โหลดข้อมูลหลัก
df = pd.read_csv("../data/footballers.csv")

# Clean ข้อมูลมูลค่าก่อน (แปลงเป็นตัวเลข และตัด NaN ในมูลค่าและตัวแปรสถิติออก)
df["value_millions"] = pd.to_numeric(df["value_millions"], errors="coerce")
df_clean = df.dropna(subset=["value_millions", "age"]).copy()

print(f"จำนวนข้อมูลพร้อมเทรน: {len(df_clean)} คน")
df_clean[["name", "overall", "age", "value_millions"]].head()

### ขั้นที่ 1 & 2: เตรียม $X, y$ และแบ่ง Train/Test

In [ ]:
# TODO: เลือก Feature 3-5 คอลัมน์ เช่น overall, age, pace, shooting, passing
features = ["overall", "age", "pace", "shooting", "passing"]
X = df_clean[features]
y = df_clean["value_millions"]

# TODO: แบ่งข้อมูล train_test_split (test_size=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

### ขั้นที่ 3: เทรนโมเดลและวัดผลเทียบกับ Baseline

In [ ]:
# TODO: สร้างและเทรน LinearRegression model ด้วย X_train, y_train
model = LinearRegression()
model.fit(X_train, y_train)

# TODO: วัดผลบน Test set ด้วย Mean Absolute Error (MAE)
y_pred = model.predict(X_test)
model_mae = mean_absolute_error(y_test, y_pred)

# Baseline: ทายทุกคนเป็นค่าเฉลี่ยของ y_train
baseline_pred = [y_train.mean()] * len(y_test)
baseline_mae = mean_absolute_error(y_test, baseline_pred)

print(f"Baseline MAE: {baseline_mae:.2f} ล้านยูโร")
print(f"Model MAE:    {model_mae:.2f} ล้านยูโร")

### ขั้นที่ 4: ทำนายราคานักเตะชุดใหม่ (Unseen Scouted Data!)
โหลดไฟล์ `../data/scouted_players.csv` ซึ่งเป็นข้อมูลนักเตะดาวรุ่งชุดใหม่ที่ไม่มีราคา แล้วให้โมเดลของเราประเมินมูลค่า

In [ ]:
# 1. โหลดข้อมูลนักเตะชุดใหม่
new_df = pd.read_csv("../data/scouted_players.csv")

# 2. ดึง Features เดียวกันกับที่ใช้เทรน
X_new = new_df[features]

# 3. ให้โมเดลทำนายมูลค่า (Predicted Value)
new_df["predicted_value_millions"] = model.predict(X_new).round(1)

# 4. แสดงผลลัพธ์การประเมินราคา
new_df[["name", "club", "age", "overall", "predicted_value_millions"]].sort_values("predicted_value_millions", ascending=False)

### คำถามทบทวนความเข้าใจ (Concept Check)
1. การทำนายมูลค่านักเตะแบบนี้ ถือเป็น Machine Learning หรือไม่? เพราะเหตุใด? (ระบุว่าเป็น Supervised หรือ Unsupervised, และเป็น Regression หรือ Classification)
2. ทำไมเราจึงต้องแบ่ง Train/Test และทำไมการนำโมเดลไป Predict บนไฟล์ `scouted_players.csv` จึงสะท้อนการใช้งานจริงของ Machine Learning?

### Check ก่อนส่ง
- [ ] Clean ข้อมูล `value_millions` ตัดค่าที่ไม่ใช่ตัวเลขออกได้ถูกต้อง
- [ ] $X$ เป็น 2D DataFrame และ $y$ คือ `value_millions`
- [ ] เทรนด้วยข้อมูล Train และวัดผล MAE บน Test set
- [ ] นำโมเดลไป predict มูลค่านักเตะใน `scouted_players.csv` ได้สำเร็จ
- [ ] ตอบคำถามทบทวนทั้ง 2 ข้อครบถ้วน